# Running ICON from Python

This notebook is a small, inspectable ICON experiment. It uses a global ICON grid, builds an analytical dry Jablonowski-Williamson 2026 initial state, advances that state with the ICON4Py dynamical core and diffusion, and plots the result directly from in-memory xarray snapshots.

The goal is not to hide ICON4Py behind a polished application. The goal is to show the main objects a model run needs:

- `config`: plain Python settings for the experiment.
- `grid`: horizontal/vertical grid information and ICON connectivity data.
- `state`: prognostic fields such as density, virtual potential temperature, Exner pressure, and wind.
- `model`: the initialized dycore/diffusion time-integration object.

The helper implementation lives in `icon4py_helper.py`; this notebook keeps only the student-facing workflow.

In [ ]:
import os
import pathlib
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    repo_dir = pathlib.Path("/content/icon4py_demo")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/ofuhrer/icon4py_demo.git",
                str(repo_dir),
            ],
            check=True,
        )
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    install_marker = pathlib.Path("/content/.icon4py_demo_deps_installed")
    if not install_marker.exists():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
            check=True,
        )
        install_marker.touch()



In [ ]:
from icon4py_helper import (
    check_config,
    create_grid,
    create_state,
    init_state,
    create_model,
    plot_field,
    plot_diagnostics,
)

## Background

The workflow uses a deliberately small API. Each call has one job:

| Call | What it does | Main output or side effect |
| --- | --- | --- |
| `check_config(config)` | Merges your config dictionary with defaults and validates values before expensive setup begins. | Returns a normalized config dictionary. |
| `create_grid(config)` | Resolves the requested ICON grid, builds ICON connectivity/geometry data, selects the backend allocator, and computes the vertical level distribution. | Returns `grid`, a dictionary with plotting coordinates, dimensions, vertical levels, and private ICON4Py grid objects. |
| `create_state(grid, config, tracers=None)` | Creates an initially empty state container that is tied to the grid/backend. | Returns `state`, a mutable run object. Its public fields become xarray objects after initialization. |
| `init_state(grid, state, "JW26", config)` | Builds the ICON4Py JW experiment config, prepares static grid support fields, allocates prognostic fields, and fills the analytical initial condition. | Mutates `state` in place and exposes the latest instant as `state["xarray"]`. |
| `plot_field(grid, field, ..., projection="flat")` | Draws one cell-centered xarray field or expression on the ICON triangular grid. If `field` is `None`, it draws only gridlines. | Displays one flat plot or one rotatable sphere. |
| `create_model(grid, state, config)` | Initializes the ICON4Py dycore/diffusion granules and assembles the time-step state. | Returns `model`. No separate explicit compile phase is required. |
| `model.step(grid, state, count=n, diagnostics=diagnostics)` | Advances the dycore/diffusion model by `n` additional timesteps from the current state. | Mutates `state`; if a diagnostics list is supplied, appends min/mean/max rows for every timestep. |

After initialization, fields such as `state["temperature"]`, `state["pressure"]`, and `state["surface_pressure"]` are xarray `DataArray`s for the current instant. Because they are ordinary xarray objects, perturbations are ordinary arithmetic: `state["temperature"] - initial_state["temperature"]`.

## Step 1: Configuration

`config` is a normal dictionary. `check_config(...)` fills defaults and validates the choices before any expensive ICON4Py setup starts. Any option you leave out is taken from the "Default if omitted" column below.

Useful options to experiment with:

| Option | Default if omitted | Typical choices | What it changes |
| --- | --- | --- | --- |
| `grid` | `"R01B01"` | `"R01B01"`, `"R02B03"`, `"R02B04"` | Horizontal resolution. `R01B01` is very coarse and good for fast testing. `R02B03` is a useful middle-resolution grid. `R02B04` is more detailed but slower. |
| `levels` | `10` | `10`, `20`, `35` | Number of vertical full levels. More levels give a richer vertical structure and more work per timestep. |
| `backend` | `"gtfn_cpu"` | `"gtfn_cpu"`, `"embedded"` | GT4Py execution backend. `gtfn_cpu` compiles CPU kernels and is the realistic local choice. `embedded` is useful for debugging but not representative for performance. |
| `dtime_seconds` | `120` | `60`, `120`, `300` | Basic model timestep length. ICON's dycore uses the shorter timestep `dtime_seconds / ndyn_substeps`. |
| `ndyn_substeps` | `5` | `2`, `5`, `10` | Number of dycore substeps inside one model timestep. More substeps cost more but can improve stability. |
| `output_frequency_steps` | `1` | `1`, `720`, `10000` | How often the helper retains private monitor snapshots. The visible saved states in this notebook are the explicit `daily_states` list below. |
| `baroclinic_amplitude` | `1.0` | `0.0`, `1.0` | Strength of the Jablonowski-Williamson perturbation. `0.0` is more symmetric; `1.0` is the default baroclinic test. |
| `log_level` | `"info"` | `"quiet"`, `"info"`, `"debug"` | How much progress information the helper prints. Use `debug` to see more setup detail. |
| `gt4py_cache_dir` | `.gt4py_cache` | path string | Where persistent GT4Py generated-code artifacts are stored. |
| `gt4py_cache_lifetime` | `"persistent"` | `"persistent"`, `"session"` | Whether compiled/generated artifacts are kept between Python sessions or only for this session. |
| `suppress_warnings` | `True` | `True`, `False` | Filters common GT4Py compile/performance and RBF interpolation warnings that are expected in this small notebook run. Set to `False` if you want to inspect them. |

`check_config(...)` also adds a derived `config["timestep_stability"]` entry. It estimates the effective mesh size from the ICON `RnBk` grid name and reports the dycore substep limit used for the stability warning.

The default below is deliberately small so students can restart and rerun the notebook quickly. It spells out the defaults explicitly; deleting any of these keys would choose the same value from `check_config(...)`.

In [ ]:
config = check_config(
    {
        "grid": "R02B03",
        "backend": "gtfn_cpu",
        "levels": 10,
        "dtime_seconds": 120,
        "ndyn_substeps": 5,
        "output_frequency_steps": 720,
        "baroclinic_amplitude": 1.0,
        "log_level": "info",
        "suppress_warnings": True,
    }
)

config

## Step 2: Grid

`create_grid(config)` reads the selected ICON grid file from `data/`, builds the ICON grid manager, and exposes a small student-friendly `grid` dictionary. The grid contains longitude/latitude arrays for plotting, horizontal dimensions, and the vertical interface distribution.

`create_state(grid, config)` creates an empty state container. `init_state(grid, state, "JW26", config)` then prepares the static grid support fields needed by the analytical initializer and fills the state with the Jablonowski-Williamson dry atmosphere.

This initialization can take time on the first run because GT4Py may compile setup kernels. Re-running with a persistent cache should be faster.

Before initializing atmospheric fields, `plot_field(grid, None, ...)` can be used to inspect only the ICON grid geometry. The flat version shows the longitude-latitude projection; the sphere version shows the same triangular cells without the longitude seam or polar projection singularity.

The spherical plots use Plotly. The helper publishes both Plotly's Jupyter MIME representation and an iframe HTML fallback so the output works across more JupyterLab, classic notebook, and browser combinations. If a local browser or notebook security setting still blocks the inline 3D view, assign the result to a variable and call `fig.write_html("sphere.html")`, then open that file directly in a browser.

In [ ]:
grid = create_grid(config)

print("grid:", {k: grid[k] for k in ["name", "kind", "dims", "num_levels", "backend"]})
print("vertical interfaces [m]:", grid["vertical_interfaces"][:4], "...", grid["vertical_interfaces"][-4:])

In [ ]:
plot_field(grid, None, title="ICON grid")

In [ ]:
plot_field(grid, None, title="ICON grid", projection="sphere")

## Step 3: Initial Condition

The helper keeps the current model instant in memory as xarray fields. `state["xarray"]` is the current instant only; it is not an ever-growing time series. If you want to keep selected output states, copy them explicitly into a list, as we do in Step 4.

`plot_field(grid, state["temperature"], ...)` plots one cell-centered xarray field by filling each triangular ICON cell with the corresponding value. The same function also accepts xarray expressions, so perturbation plots are direct: `plot_field(grid, state["temperature"] - initial_state["temperature"])`.

The default `projection="flat"` uses longitude and latitude axes. `projection="sphere"` draws the same ICON triangles on a rotatable Plotly sphere. The flat plot uses longitude and latitude as Cartesian axes; near the poles this projection is singular, so small white polar gaps are a plotting artifact, not missing grid cells.

In [ ]:
state = create_state(grid, config, tracers=None)
print("state fields:", [k for k in ["rho", "theta_v", "exner", "vn", "w"] if state[k] is not None])

In [ ]:
init_state(grid, state, "JW26", config)

print("current xarray sizes:", dict(state["xarray"].sizes))
plot_field(
    grid,
    state["temperature"],
    title="Initial Condition (Temperature)",
    colorbar_label="K",
)

initial_state = state["xarray"].copy(deep=True)

In [ ]:
plot_field(
    grid,
    state["temperature"],
    title="Initial Condition (Temperature)",
    projection="sphere",
)

## Step 4: Integrate

`create_model(grid, state, config)` is where the actual ICON4Py dycore/diffusion granules and time-step state are initialized. There is no separate compile button: GT4Py compilation is lazy, so creating the model and the first call to `model.step(...)` may still compile kernels.

`count` is the number of additional timesteps to take from the current state. It is not the absolute timestep number. Here we compute how many timesteps make one day, then call `model.step(...)` once per simulated day. After each day, we copy the current xarray state into `daily_states`. Diagnostics are a separate list and are only collected because we pass `diagnostics=diagnostics` into `model.step(...)`.

In [ ]:
model = create_model(grid, state, config)

diagnostics = []
daily_states = [initial_state]
timesteps_per_day = round(24 * 60 * 60 / config["dtime_seconds"])
days_to_run = 8

for day in range(1, days_to_run + 1):
    print(f"integrating day {day}: {timesteps_per_day} timesteps")
    model.step(grid, state, count=timesteps_per_day, diagnostics=diagnostics)
    daily_states.append(state["xarray"].copy(deep=True))

print("captured daily states:", len(daily_states))
print("recorded diagnostic timesteps:", len(diagnostics))

In [ ]:
plot_field(
    grid,
    daily_states[-1]["temperature"],
    title=f"Temperature after day {days_to_run}",
    colorbar_label="K",
)
plot_field(
    grid,
    daily_states[-1]["temperature"] - daily_states[0]["temperature"],
    title=f"Temperature perturbation after day {days_to_run}",
    colorbar_label="K",
)
plot_diagnostics(diagnostics, fields=["temperature", "rho", "exner"])